# Reasons Analysis

Full pipeline for analysing ICF (Interval Constraint Function) reasons and anti-reasons
from MCR experiment checkpoints:

1. Load checkpoint from directory
2. Extract test samples and compute sigmas
3. Calculate costs (parallel, incremental cache)
4. Compute robustness per ICF and per sample
5. Visualise cost distributions, robustness, and ICF constraints

**Robustness formula (IEEE CAI 2026)**:
$$r(C, x) = 1 - \frac{\max_{\text{ICF} \in AR_{C,y}} \text{cost}_x(\text{ICF})}{|\text{features}|}$$

In [1]:
from pathlib import Path
from etl.loader import etl_from_dir, list_checkpoints

RESULTS_DIR = Path("results")

# List available checkpoints
checkpoints = list_checkpoints(RESULTS_DIR)
print(f"Found {len(checkpoints)} checkpoints:")
for i, cp in enumerate(checkpoints):
    print(f"  [{i}] {cp.relative_to(RESULTS_DIR)}")

# Select checkpoint (edit index or set CHECKPOINT_DIR directly)
CHECKPOINT_DIR = checkpoints[0] if checkpoints else None
print(f"\nUsing: {CHECKPOINT_DIR}")

Found 157 checkpoints:
  [0] checkpoints/amyloid_pos_vs_neg_fs3_fold1/workers_6/class_1
  [1] res_ablation/ann-thyroid/workers_16/class_1.0/ablation_R0_NR0_GP1_BP1
  [2] res_ablation/ann-thyroid/workers_16/class_1.0/ablation_R0_NR1_GP1_BP1
  [3] res_ablation/ann-thyroid/workers_16/class_1.0/ablation_R1_NR0_GP1_BP1
  [4] res_ablation/ann-thyroid/workers_16/class_1.0/ablation_R1_NR1_GP0_BP1
  [5] res_ablation/ann-thyroid/workers_16/class_1.0/ablation_R1_NR1_GP1_BP0
  [6] res_ablation/ann-thyroid/workers_16/class_1.0/all_features
  [7] res_ablation/ann-thyroid/workers_16/class_2.0/ablation_R0_NR1_GP1_BP1
  [8] res_ablation/ann-thyroid/workers_16/class_2.0/ablation_R1_NR0_GP1_BP1
  [9] res_ablation/ann-thyroid/workers_16/class_2.0/ablation_R1_NR1_GP0_BP1
  [10] res_ablation/ann-thyroid/workers_16/class_2.0/ablation_R1_NR1_GP1_BP0
  [11] res_ablation/ann-thyroid/workers_16/class_2.0/all_features
  [12] res_ablation/ann-thyroid/workers_16/class_3.0/ablation_R0_NR1_GP1_BP1
  [13] res_ablation

In [2]:
# Load checkpoint
db = etl_from_dir(CHECKPOINT_DIR, verbose=True)
dataset_name = db['_dataset_name']

Loading checkpoint: results/checkpoints/amyloid_pos_vs_neg_fs3_fold1/workers_6/class_1
Dataset: amyloid_pos_vs_neg_fs3_fold1
  DB00 (data): 66 keys
  DB01 (candidates): 0 keys
  DB02 (reasons): 18 keys
  DB03 (non_reasons): 135 keys
  DB04 (candidate_anti_reasons): 1164 keys
  DB05 (anti_reasons): 68 keys
  DB06 (good_profiles): 18 keys
  DB07 (bad_profiles): 135 keys
  DB08 (preferred_reasons): 20 keys
  DB09 (anti_reason_profiles): 68 keys
  DB10 (workers_report): 4043 keys
Loaded 11 databases.


In [3]:
# Extract test samples and compute split-Gaussian sigmas
from cost_function import cal_sigmas
from etl.reasons_analysis import extract_test_samples

tests_sample, X_test, test_ids, feature_names = extract_test_samples(db)

X_train = db["data"]["TRAINING_SET"].get("value_json", db["data"]["TRAINING_SET"])["X_train"]
sigmas_all = cal_sigmas(X_train, X_test, feature_names, test_ids=test_ids)

print(f"Extracted {len(test_ids)} test samples")
print(f"Features: {len(feature_names)}")

Extracted 20 test samples
Features: 11


In [4]:
# Calculate costs for all ICFs (parallel, incremental)
from cost_function import cost_function
from redis_helpers.icf import bitmap_to_icf
import pandas as pd

reason_types = ['reasons', 'non_reasons', 'anti_reasons']

print("Calculating costs for all reason types...")
print("=" * 80)

cost_records = []
eu_raw = db["data"]['EU']
eu = eu_raw.get("value_json", eu_raw) if isinstance(eu_raw, dict) and "value_json" in eu_raw else eu_raw

# Expected bitmap length derived from EU
expected_bitmap_len = sum(len(v) - 1 for v in eu.values()) + len(eu)
print(f"Expected bitmap length: {expected_bitmap_len}")

for reason_type in reason_types:
    if reason_type not in db or not db[reason_type]:
        print(f"  {reason_type}: not found or empty, skipping")
        continue
    bitmaps = [k for k in db[reason_type].keys() if set(k) <= {'0', '1'}]
    print(f"  {reason_type}: {len(bitmaps)} ICFs")
    for bitmap_string in bitmaps:
        try:
            icf = bitmap_to_icf(bitmap_string, eu)
        except Exception as e:
            print(f"    skip bitmap (len={len(bitmap_string)}): {e}")
            continue
        for sample_id in test_ids:
            sample_data = tests_sample[sample_id]
            sample_data.setdefault(reason_type, {})
            cost = cost_function(
                sample=sample_data['features'],
                sigmas=sigmas_all[sample_id],
                icf=icf
            )
            sample_data[reason_type][bitmap_string] = {'icf': icf, 'cost': cost}
            cost_records.append({
                'reason_type': reason_type,
                'bitmap': bitmap_string,
                'sample_id': sample_id,
                'cost': cost,
                'icf': icf,
            })

cost_df = pd.DataFrame(cost_records)
print(f"\nTotal costs calculated: {len(cost_df)}")
print(cost_df.groupby('reason_type')['cost'].describe())

Calculating costs for all reason types...
Expected bitmap length: 754
  reasons: 18 ICFs
  non_reasons: 135 ICFs
  anti_reasons: 68 ICFs

Total costs calculated: 4420
               count      mean       std       min       25%       50%  \
reason_type                                                              
anti_reasons  1360.0  1.685846  0.618699  0.571246  1.284682  1.590297   
non_reasons   2700.0  1.856592  0.687163  0.571246  1.365667  1.771062   
reasons        360.0  7.361001  1.665273  3.460621  5.805320  8.046590   

                   75%        max  
reason_type                        
anti_reasons  2.050881   4.607424  
non_reasons   2.272464   4.607424  
reasons       8.283417  10.378048  


## Robustness Calculation

Per-sample robustness using anti-reasons only (as per the IEEE CAI 2026 paper).

In [5]:
from etl.reasons_analysis import (
    calculate_robustness_per_bitmap,
    calculate_all_samples_robustness,
    print_robustness_statistics
)

num_features = len(feature_names)

print(f"{'='*80}")
print(f"  ROBUSTNESS CALCULATION (Anti-Reasons Only)")
print(f"{'='*80}\n")
print(f"  Number of features: {num_features}")
print(f"  Formula: r(C,x) = 1 - max{{ICF ∈ AR{{C,y}}}} cost_x(ICF) / |features|")

anti_reasons_df = cost_df[cost_df['reason_type'] == 'anti_reasons'].copy()

if len(anti_reasons_df) == 0:
    print("\nWARNING: No anti_reasons found in cost data.")
else:
    bitmap_robustness = calculate_robustness_per_bitmap(anti_reasons_df, num_features=num_features)

    print(f"\n  Total anti-reason ICFs: {len(bitmap_robustness)}")
    print(f"  Max cost:      {bitmap_robustness['max_cost'].max():.6f}")
    print(f"  Mean max cost: {bitmap_robustness['max_cost'].mean():.6f}")
    print(f"  Min max cost:  {bitmap_robustness['max_cost'].min():.6f}")

    sample_robustness_df = calculate_all_samples_robustness(
        cost_df=cost_df,
        num_features=num_features,
        tests_sample=tests_sample,
        test_ids=test_ids,
        verbose=True
    )

    print_robustness_statistics(sample_robustness_df)

    Path('results').mkdir(exist_ok=True)
    sample_robustness_df.to_csv('results/sample_robustness.csv', index=False)
    print(f"\nResults saved to: results/sample_robustness.csv")

  ROBUSTNESS CALCULATION (Anti-Reasons Only)

  Number of features: 11
  Formula: r(C,x) = 1 - max{ICF ∈ AR{C,y}} cost_x(ICF) / |features|

  Total anti-reason ICFs: 68
  Max cost:      4.607424
  Mean max cost: 2.742268
  Min max cost:  1.809414
Processing 20 samples...
 All 20 samples processed


  SAMPLE-LEVEL ROBUSTNESS ANALYSIS

 Dataset Overview:
   • Total samples:        20
   • Correct predictions:  0 (0.0%)
   • Incorrect predictions: 20 (100.0%)

 Robustness Statistics (All Samples):
   • Mean:      0.718764 +/- 0.060183
   • Median:    0.717231
   • Min/Max:   0.581143 / 0.817164
   • Range:     0.236020
   • IQR:       0.073545 (Q1=0.684260, Q3=0.757805)

 Robustness by Prediction Correctness:
--------------------------------------------------------------------------------

--------------------------------------------------------------------------------

Results saved to: results/sample_robustness.csv


In [6]:
# Robustness distribution visualisations (plotly)
from etl.reasons_analysis import create_robustness_visualizations

if 'sample_robustness_df' in locals() and len(sample_robustness_df) > 0:
    fig_main, fig_quartiles = create_robustness_visualizations(
        sample_robustness_df=sample_robustness_df,
        dataset_name=dataset_name
    )
    if fig_main is not None:
        fig_main.show()
    if fig_quartiles is not None:
        fig_quartiles.show()
else:
    print("sample_robustness_df not available. Run previous cell first.")

In [7]:
# Feature-level cost distribution visualisations
# Skipped for large datasets to avoid memory issues
print("Cost distribution visualization skipped (use sample-level analysis above).")

Cost distribution visualization skipped (use sample-level analysis above).


In [8]:
# Select a sample for detailed ICF visualisation
sample_id = test_ids[0]
print(f"Analysing sample: {sample_id}")
print(f"Features: {feature_names[:10]}{'...' if len(feature_names) > 10 else ''}")

Analysing sample: amyloid_pos_vs_neg_fs3_fold1_1_4
Features: ['GENDER', 'AGE_AT_MRI', 'EDU', 'APOE4', 'MMSE', 'ADAS13', 'MEM', 'EF', 'LAN', 'VS']...


In [9]:
# Visualise per-feature ICF constraints as bar charts (tabular data)
from etl.reasons_analysis import visualize_sample_with_icf

# Reason ICF
fig = visualize_sample_with_icf(sample_id, tests_sample, feature_names, reason_type='reasons')
if fig is not None:
    fig.show()


REASONS Analysis for Sample amyloid_pos_vs_neg_fs3_fold1_1_4:
  Cost: 6.917649
  Constrained features: 7/11
  Temporal intervals: 2


In [10]:
# Anti-Reason ICF
fig = visualize_sample_with_icf(sample_id, tests_sample, feature_names, reason_type='anti_reasons')
if fig is not None:
    fig.show()


ANTI-REASONS Analysis for Sample amyloid_pos_vs_neg_fs3_fold1_1_4:
  Cost: 1.462510
  Constrained features: 11/11
  Temporal intervals: 1


In [11]:
# Reason vs Anti-Reason comparison
from etl.reasons_analysis import visualize_sample_comparison

fig = visualize_sample_comparison(sample_id, tests_sample, feature_names)
if fig is not None:
    fig.show()


Comparison for Sample amyloid_pos_vs_neg_fs3_fold1_1_4:

  REASON:
    Constrained features: 7
    Temporal intervals: 2
  ANTI-REASON:
    Constrained features: 11
    Temporal intervals: 1


In [12]:
# Smooth corridor comparison
from etl.reasons_analysis import visualize_sample_comparison_smooth

fig = visualize_sample_comparison_smooth(sample_id, tests_sample, feature_names)
if fig is not None:
    fig.show()


Smooth Corridor Comparison for Sample amyloid_pos_vs_neg_fs3_fold1_1_4:

  REASON CORRIDOR:
    Cost: 6.917649
    Constrained features: 7/11
    Corridor width: avg=0.206

  ANTI-REASON CORRIDOR:
    Cost: 1.462510
    Constrained features: 11/11
    Corridor width: avg=0.153


In [13]:
# Anti-reason corridor coloured by constraint satisfaction
from etl.reasons_analysis import visualize_anti_reason_corridor

fig = visualize_anti_reason_corridor(sample_id, tests_sample, feature_names)
if fig is not None:
    fig.show()


Anti-Reason Constraint Analysis for Sample amyloid_pos_vs_neg_fs3_fold1_1_4:

  CONSTRAINT SATISFACTION:
    Cost: 1.462510
    Total features: 11
    Constrained features: 11
    Unconstrained features: 0
     Within constraints: 2/11 (18.2%)
     Constraint violations: 9/11
    Violated features: GENDER, AGE_AT_MRI, MMSE, 13, MEM
      ... and 4 more


In [14]:
# Robustness per ICF (bitmap-level)
if 'anti_reasons_df' in locals() and len(anti_reasons_df) > 0:
    print("="*80)
    print("ROBUSTNESS PER ICF (Anti-Reasons Only)")
    print("="*80)
    print(f"\nTotal anti-reason ICFs: {len(bitmap_robustness)}")
    print("\nTop 10 by max_cost (hardest anti-reasons):")
    print(bitmap_robustness[['max_cost', 'robustness', 'mean_cost', 'n_samples']].head(10).to_string())

    print("\nBottom 5 robustness (most vulnerable):")
    ar_lowest = bitmap_robustness.nsmallest(5, 'robustness')
    print(ar_lowest[['max_cost', 'robustness', 'mean_cost', 'n_samples']].to_string())

    bitmap_robustness.to_csv('results/anti_reasons_robustness.csv', index=False)
    print("\nSaved to: results/anti_reasons_robustness.csv")

ROBUSTNESS PER ICF (Anti-Reasons Only)

Total anti-reason ICFs: 68

Top 10 by max_cost (hardest anti-reasons):
    max_cost  robustness  mean_cost  n_samples
40  4.607424    0.581143   2.424930         20
37  4.602340    0.581605   2.421617         20
27  4.577494    0.583864   2.885117         20
1   4.188320    0.619244   2.296518         20
38  4.140282    0.623611   2.356883         20
53  3.986378    0.637602   2.257032         20
36  3.939597    0.641855   2.198739         20
43  3.502583    0.681583   1.866528         20
54  3.463320    0.685153   2.008998         20
42  3.418605    0.689218   1.885303         20

Bottom 5 robustness (most vulnerable):
    max_cost  robustness  mean_cost  n_samples
40  4.607424    0.581143   2.424930         20
37  4.602340    0.581605   2.421617         20
27  4.577494    0.583864   2.885117         20
1   4.188320    0.619244   2.296518         20
38  4.140282    0.623611   2.356883         20

Saved to: results/anti_reasons_robustness.csv
